In [ ]:
%load_ext autoreload
%autoreload 2

# Imports

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_google_vertexai import ChatVertexAI
import matplotlib.pyplot as plt

from wsd.load_data import load_data
from wsd.models import BinaryWSD, ClusterByMeaningModel, DummyComparator
from linpub.metrics import accuracy

from lineval.utils import (load_hub_docs,
                           check_assign_docs,
                           OpenAiWordSenseComparatorv4,
                           ClusterByMeaningModelv4,
                           annotator_filter,
                           AnnotationSerializerv2,
                           AnnotationFactoryv2,
                           get_unique_senses,
                           plot_cluster_accuracy
)

from linalgo.hub.client import LinalgoClient
from linalgo.annotate.models import Corpus, Document, Annotation, Target, Selector, Annotator
from lineval.utils import Body
from datetime import datetime
from collections import defaultdict
from dotenv import load_dotenv
import os

# Testing linhub loading capacity

In [ ]:
load_dotenv()
token = os.getenv('LINHUB_TOKEN')
url = "https://linhub.api.linalgo.com/v1"
client = LinalgoClient(token, url)
task_id = 'd3ce7764-eb85-4999-b965-c028f539ee33'
wsd_task = client.get_task(task_id, verbose=True)

In [ ]:
jack = wsd_task.annotators[1]
jack.__dict__

In [ ]:
# Use create_...


#1. creating the annotator
# THIS WAS RUN ONCE ALREADY

# gold_annotator = Annotator(name="gold",
#                            task = wsd_task,
#                            owner=jack.owner,
#                            model="MACHINE")
# gold_annotator.__dict__
# client.create_annotator(gold_annotator)

In [ ]:
client.get_annotators(wsd_task)

# Loading candidates

In [ ]:
#loading candidates
X, y = load_data(lang='fr')

k = len(X)
X_test, y_test = X[:k], y[:k]
len(X_test), len(y_test)

In [ ]:
# canidates to corpus
semcor_corpus = Corpus(name='Semcor')

grouped_X = defaultdict(list)
for i,row in enumerate(X_test):
    row.lemma_meaning = y_test[i]
    grouped_X[(row.lemma, row.pos)].append(row)
items = grouped_X.items()
docs = []
for g, cands in items:
    contexts = "\n".join([anno.context for anno in cands])
    doc = Document(content=contexts,
                   corpus=semcor_corpus
                   )
    doc_annos = []
    for c in cands:
        anno = Annotation(document=doc,
                          entity=c.lemma_meaning,
                          body=Body(text=c.text, context=c.context),
                          task="task",
                          annotator="none",
                          target={},
                          created=datetime.now())
        doc_annos.append(anno)
    doc.annotations = set(doc_annos)
    docs.append(doc)

semcor_corpus.documents = docs
X_docs = semcor_corpus.documents
len(X_docs)

In [ ]:
semcor_corpus.__dict__

In [ ]:
corpus = client.get_corpus(wsd_task.corpora[0].id)
corpus.__dict__

In [ ]:
post_url = url + f"/corpora/"
jack_org = "acf7a1aa-ec18-4fa2-a981-a756bc6e6af2"
payload = {
  "id": semcor_corpus.id,
  "name": semcor_corpus.name,
  "description": "None",
  "created": "2025-02-17T03:38:29.371Z",
  "edited": "2025-02-17T03:38:29.371Z",
  "is_private": True,
  "organization": jack_org,
}

In [ ]:
client.post(url = post_url, data= semcor_corpus)

## printing

In [ ]:
example_ano = list(Semcor_corpus.documents[0].annotations)[0]

Semcor_corpus.documents[0].__dict__, example_ano.__dict__, example_ano.annotator.__dict__

# Creating gold

In [ ]:
gold_annotator = Annotator(name="gold",
                           owner="jack",
                           model="MACHINE")
gold_annotator.__dict__

In [ ]:
# ONLY RUN ONCE!!!

for doc in X_docs:
    gold_annos = [anno for anno in doc.annotations if anno.annotator.name == None]
    for anno in gold_annos:
        a = Annotation(entity = anno.entity,
                       body=anno.body,
                       document=doc,
                       annotator = gold_annotator,
                       target = anno.target,
                       )
        doc.annotations.add(a)


In [ ]:
list(X_docs[0].annotations)[0].__dict__

# Predicting

## Base predict (avoid)

In [ ]:
load_dotenv()
api_key = os.getenv("OPEN_AI_API_KEY")
comp_1 = OpenAiWordSenseComparatorv4(api_key=api_key, openai_model="gpt-4o", thought_process=False)
model_1 = ClusterByMeaningModelv4(word_sense_comparator=comp_1)

In [ ]:
y_pred_docs = model_1.predict(X_docs)

In [ ]:
dummy_1 = Annotator(name="dummy_1",
                           owner="jack",
                           model="MACHINE")
dummy_0 = Annotator(name="dummy_0",
                    owner="jack",
                    model="MACHINE")


for doc in X_docs:
    gold_annos = [anno for anno in doc.annotations if anno.annotator.name == "gold"]
    i = 0
    for anno in gold_annos:
        a = Annotation(entity = "0",
                       body=anno.body,
                       document=doc,
                       annotator = dummy_0,
                       target = anno.target,
                       )
        b = Annotation(entity = str(i),
                       body=anno.body,
                       document=doc,
                       annotator = dummy_1,
                       target = anno.target,
                       )
        i += 1
        doc.annotations.add(a)
        doc.annotations.add(b)

## batch predict (use)

In [ ]:
# CODE HERE

# visualisations

In [ ]:


doc_len = [len(list(doc.annotations)) for doc in X_docs]

plt.plot(doc_len)
plt.title("Number of annotations per document")
plt.show()

In [ ]:
X_docs_sorted = sorted(X_docs, key=lambda x: len(list(x.annotations)), reverse=False)

sorted_doc_len = [len(list(doc.annotations)) for doc in X_docs_sorted]

plt.plot(sorted_doc_len)
plt.title("Number of annotations per document (sorted)")
plt.show()

In [ ]:
sorted_doc_len[0]

In [ ]:
# removing docs with only 2 annotations(Semcor and gold)
X_docs_filtered = [doc for doc in X_docs if len(list(doc.annotations)) > 2]
X_docs_filtered.sort(key=lambda x: len(list(x.annotations)), reverse=False)
len(X_docs_filtered)

In [ ]:
plt.plot([len(list(doc.annotations))/5 for doc in X_docs_filtered])
plt.title("Number of annotations per ML annotated document")
plt.show()

In [ ]:
nb_annos = set(int(len(list(doc.annotations))/5) for doc in X_docs_filtered)
plt.hist(nb_annos, bins = 23 )
plt.ylabel("Number of annotations per document")
plt.show
#sort the set

In [ ]:
nb_annos = list(sorted(nb_annos))
len(nb_annos), nb_annos

In [ ]:
accs = []
dummy_0_accs = []
dummy_1_accs = []

for x_val in nb_annos:
    y = []
    y_pred = []
    dummy_0_y_pred = []
    dummy_1_y_pred = []

    for doc in X_docs_filtered:
        if len(list(doc.annotations)) == x_val * 5:
            for anno in sorted(doc.annotations, key=lambda x: x.body.context):
                sense = hash((doc, anno.entity.id))
                if anno.annotator.name == "gold":
                    y.append(sense)
                if anno.annotator.name == "gpt-4o_tp_False":
                    y_pred.append(sense)
                if anno.annotator.name == "dummy_0":
                    dummy_0_y_pred.append(sense)
                if anno.annotator.name == "dummy_1":
                    dummy_1_y_pred.append(sense)

    if len(y) > 0:
        accs.append(accuracy(y_pred, y))
        dummy_0_accs.append(accuracy(dummy_0_y_pred, y))
        dummy_1_accs.append(accuracy(dummy_1_y_pred, y))

plt.plot(nb_annos, accs, label="GPT-4o")
plt.plot(nb_annos, dummy_0_accs, label="Dummy 0", linestyle="--")
plt.plot(nb_annos, dummy_1_accs, label="Dummy 1", linestyle=":")

plt.title("Accuracy per Number of Annotations")
plt.xlabel("Number of Annotations")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
nb_clust = []
for doc in X_docs_filtered:
    gold_annos = [anno for anno in doc.annotations if anno.annotator.name == "gold"]
    local_clusters = len(set( anno.entity.id for anno in gold_annos))
    nb_clust.append(local_clusters)
cluster_sizes = list(sorted(set(nb_clust)))
cluster_sizes

In [ ]:
accs=[]
dummy_0_accs = []
dummy_1_accs = []
for x_val in cluster_sizes:
    y=[]
    y_pred=[]
    dummy_0_y_pred = []
    dummy_1_y_pred = []
    for doc in X_docs_filtered:
        gold_annos = [anno for anno in doc.annotations if anno.annotator.name == "gold"]
        local_clusters = len(set( anno.entity.id for anno in gold_annos))
        if local_clusters == x_val:
            for anno in sorted(doc.annotations, key=lambda x: x.body.context):
                sense = hash((doc, anno.entity.id))
                if anno.annotator.name == "gold":
                    y.append(sense)
                if anno.annotator.name == "gpt-4o_tp_False":
                    y_pred.append(sense)
                if anno.annotator.name == "dummy_0":
                    dummy_0_y_pred.append(sense)
                if anno.annotator.name == "dummy_1":
                    dummy_1_y_pred.append(sense)
    if len(y) > 0:
        accs.append(accuracy(y_pred, y))
        dummy_0_accs.append(accuracy(dummy_0_y_pred, y))
        dummy_1_accs.append(accuracy(dummy_1_y_pred, y))

import matplotlib.pyplot as plt

accs = []
dummy_0_accs = []
dummy_1_accs = []

for x_val in cluster_sizes:
    y = []
    y_pred = []
    dummy_0_y_pred = []
    dummy_1_y_pred = []

    for doc in X_docs_filtered:
        gold_annos = [anno for anno in doc.annotations if anno.annotator.name == "gold"]
        local_clusters = len(set(anno.entity.id for anno in gold_annos))

        if local_clusters == x_val:
            for anno in sorted(doc.annotations, key=lambda x: x.body.context):
                sense = hash((doc, anno.entity.id))
                if anno.annotator.name == "gold":
                    y.append(sense)
                if anno.annotator.name == "gpt-4o_tp_False":
                    y_pred.append(sense)
                if anno.annotator.name == "dummy_0":
                    dummy_0_y_pred.append(sense)
                if anno.annotator.name == "dummy_1":
                    dummy_1_y_pred.append(sense)

    if len(y) > 0:
        accs.append(accuracy(y_pred, y))
        dummy_0_accs.append(accuracy(dummy_0_y_pred, y))
        dummy_1_accs.append(accuracy(dummy_1_y_pred, y))

# Plot accuracy vs. number of clusters
plt.plot(cluster_sizes, accs, label="GPT-4o")
plt.plot(cluster_sizes, dummy_0_accs, label="Dummy 0", linestyle="--")
plt.plot(cluster_sizes, dummy_1_accs, label="Dummy 1", linestyle=":")

plt.title("Accuracy per Number of Clusters")
plt.xlabel("Number of Clusters")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
# Dummy 1 seems very good, let's check the average number of annotations per cluster


In [ ]:
plot_cluster_accuracy(X_docs_filtered[:200])

In [ ]:
plot_cluster_accuracy(X_docs_filtered[:400])

In [ ]:
plot_cluster_accuracy(X_docs_filtered[:450])

In [ ]:
plot_cluster_accuracy(X_docs_filtered)

In [ ]:
#last 50 docs
plot_cluster_accuracy(X_docs_filtered[-50:])